In [1]:
"""
BridgeDTI + ESM — Regression Version (FIXED GCN OUTPUT DIMENSION)
Fixed: GCN output layer now correctly projects to outSize
"""

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

warnings.filterwarnings('ignore', message='.*DEPRECATION WARNING.*')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # Отключает все предупреждения RDKit
print("All libraries imported successfully")


# ==========================================
# 0. PATH CONFIGURATION
# ==========================================
print("Setting up paths...")

BASE_DIR = os.path.expanduser("~/project") if not os.getenv("COLAB_GPU") else "/content/project"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "cache"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "results"), exist_ok=True)

print(f"Base project directory: {BASE_DIR}")

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'data_path': os.path.join(BASE_DIR, "data", "human_kinase_chembl_Kd_Ki (1).csv"),
    'esm_path': os.path.join(BASE_DIR, "cache", "esm_embeddings.npy"),
    'target_column': 'pchembl_value',
    'smiles_column': 'smiles',
    'protein_id_column': 'uniprot_id',
    'outSize': 128,
    'cHiddenSizeList': [64, 32],
    'fHiddenSizeList': [512, 256],
    'fSize': 1024,
    'cSize': 8422,
    'gcnHiddenSizeList': [],  # FIX: Empty to avoid bottleneck
    'fcHiddenSizeList': [64, 32],
    'nodeNum': 32,
    'resnet': True,
    'hdnDropout': 0.1,
    'fcDropout': 0.2,
    'batch_size': 32,
    'epochs': 150,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'patience': 15,
    'esm_dim': 1280,
    'esm_agg': 'mean',
    'cold_protein_split': False,
    'log_file': os.path.join(BASE_DIR, "results", "experiments_log.csv"),
    'model_save_path': os.path.join(BASE_DIR, "results", "best_model.pth"),
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"Device: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ==========================================
# 1. NN LAYERS (FIXED GCN)
# ==========================================

class TextEmbedding(nn.Module):
    def __init__(self, embedding, dropout=0.3, freeze=False, name='textEmbedding'):
        super(TextEmbedding, self).__init__()
        self.name = name
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding, dtype=torch.float32).detach(), freeze=freeze
        )
        self.dropout1 = nn.Dropout2d(p=dropout/2)
        self.dropout2 = nn.Dropout(p=dropout/2)
        self.p = dropout
        
    def forward(self, x):
        if self.p > 0:
            x = self.dropout2(self.dropout1(self.embedding(x)))
        else:
            x = self.embedding(x)
        return x


class MLP(nn.Module):
    def __init__(self, inSize, outSize, hiddenList=[], dropout=0.0, 
                 bnEveryLayer=False, dpEveryLayer=False, outBn=False, 
                 outAct=False, outDp=False, name='MLP', actFunc=nn.ReLU):
        super(MLP, self).__init__()
        self.name = name
        self.inSize = inSize
        self.outSize = outSize
        hiddens, bns = [], []
        for i, os in enumerate(hiddenList):
            hiddens.append(nn.Sequential(nn.Linear(inSize, os)))
            bns.append(nn.BatchNorm1d(os))
            inSize = os
        bns.append(nn.BatchNorm1d(outSize))
        self.actFunc = actFunc()
        self.dropout = nn.Dropout(p=dropout)
        self.hiddens = nn.ModuleList(hiddens)
        self.bns = nn.ModuleList(bns)
        self.out = nn.Linear(inSize, outSize)
        self.bnEveryLayer = bnEveryLayer
        self.dpEveryLayer = dpEveryLayer
        self.outBn = outBn
        self.outAct = outAct
        self.outDp = outDp
        
    def forward(self, x):
        for h, bn in zip(self.hiddens, self.bns):
            x = h(x)
            if self.bnEveryLayer:
                x = bn(x) if len(x.shape) == 2 else bn(x.transpose(-1, -2)).transpose(-1, -2)
            x = self.actFunc(x)
            if self.dpEveryLayer:
                x = self.dropout(x)
        x = self.out(x)
        if self.outBn:
            x = self.bns[-1](x) if len(x.shape) == 2 else self.bns[-1](x.transpose(-1, -2)).transpose(-1, -2)
        if self.outAct:
            x = self.actFunc(x)
        if self.outDp:
            x = self.dropout(x)
        return x


class GCN(nn.Module):
    def __init__(self, inSize, outSize, hiddenSizeList=[], dropout=0.0,
                 bnEveryLayer=False, dpEveryLayer=False, outBn=False,
                 outAct=False, outDp=False, resnet=False, name='GCN', actFunc=nn.ReLU):
        super(GCN, self).__init__()
        self.name = name
        self.inSize = inSize
        self.outSize = outSize
        
        # FIX: Ensure output layer projects to outSize
        hiddens, bns = [], []
        curr_size = inSize
        for i, os in enumerate(hiddenSizeList):
            hiddens.append(nn.Sequential(nn.Linear(curr_size, os)))
            bns.append(nn.BatchNorm1d(os))
            curr_size = os
        
        # Output layer: last hidden size -> outSize
        bns.append(nn.BatchNorm1d(outSize))
        self.actFunc = actFunc()
        self.dropout = nn.Dropout(p=dropout)
        self.hiddens = nn.ModuleList(hiddens)
        self.bns = nn.ModuleList(bns)
        self.out = nn.Linear(curr_size, outSize)  # FIX: Explicitly project to outSize
        self.bnEveryLayer = bnEveryLayer
        self.dpEveryLayer = dpEveryLayer
        self.outBn = outBn
        self.outAct = outAct
        self.outDp = outDp
        self.resnet = resnet
        
    def forward(self, x, L):
        for h, bn in zip(self.hiddens, self.bns):
            a = h(torch.matmul(L, x))
            if self.bnEveryLayer:
                if len(L.shape) == 3:
                    a = bn(a.transpose(1, 2)).transpose(1, 2)
                else:
                    a = bn(a)
            a = self.actFunc(a)
            if self.dpEveryLayer:
                a = self.dropout(a)
            if self.resnet and a.shape == x.shape:
                a += x
            x = a
        
        # FIX: Output layer always projects to outSize
        a = self.out(torch.matmul(L, x))
        if self.outBn:
            if len(L.shape) == 3:
                a = self.bns[-1](a.transpose(1, 2)).transpose(1, 2)
            else:
                a = self.bns[-1](a)
        if self.outAct:
            a = self.actFunc(a)
        if self.outDp:
            a = self.dropout(a)
        if self.resnet and a.shape == x.shape:
            a += x
        return x


class TextCNN(nn.Module):
    def __init__(self, featureSize, filterSize, contextSizeList, 
                 reduction='pool', actFunc=nn.ReLU, bn=False, ln=False, name='textCNN'):
        super(TextCNN, self).__init__()
        moduleList = []
        bns, lns = [], []
        for i in range(len(contextSizeList)):
            moduleList.append(
                nn.Conv1d(in_channels=featureSize, out_channels=filterSize, 
                         kernel_size=contextSizeList[i], padding=contextSizeList[i]//2)
            )
            bns.append(nn.BatchNorm1d(filterSize))
            lns.append(nn.LayerNorm(filterSize))
        if bn:
            self.bns = nn.ModuleList(bns)
        if ln:
            self.lns = nn.ModuleList(lns)
        self.actFunc = actFunc()
        self.conv1dList = nn.ModuleList(moduleList)
        self.reduction = reduction
        self.bn = bn
        self.ln = ln
        self.name = name
        
    def forward(self, x):
        x = x.transpose(1, 2)
        x = [conv(x).transpose(1, 2) for conv in self.conv1dList]
        if self.bn:
            x = [b(i.transpose(1, 2)).transpose(1, 2) for b, i in zip(self.bns, x)]
        elif self.ln:
            x = [l(i) for l, i in zip(self.lns, x)]
        x = [self.actFunc(i) for i in x]
        if self.reduction == 'pool':
            x = [F.adaptive_max_pool1d(i.transpose(1, 2), 1).squeeze(dim=2) for i in x]
            return torch.cat(x, dim=1)
        elif self.reduction == 'None':
            return x


# ==========================================
# 2. BRIDGEDTI MODEL
# ==========================================

class DTI_Bridge_Regression(nn.Module):
    def __init__(self, outSize, cHiddenSizeList, fHiddenSizeList, 
                 fSize=1024, cSize=8422, gcnHiddenSizeList=[], fcHiddenSizeList=[], 
                 nodeNum=32, resnet=True, hdnDropout=0.1, fcDropout=0.2, 
                 device=torch.device('cuda'), useFeatures=None, 
                 esm_dim=1280, esm_agg='mean'):
        super(DTI_Bridge_Regression, self).__init__()
        
        if useFeatures is None:
            useFeatures = {"kmers": True, "pSeq": True, "FP": True, "dSeq": True}
        
        self.device = device
        self.nodeNum = nodeNum
        self.useFeatures = useFeatures
        self.esm_agg = esm_agg
        self.cSize = cSize
        self.esm_dim = esm_dim
        self.outSize = outSize
        
        self.nodeEmbedding = TextEmbedding(
            torch.tensor(np.random.normal(size=(max(nodeNum, 0), outSize)), dtype=torch.float32), 
            dropout=hdnDropout, name='nodeEmbedding'
        ).to(device)
        
        self.amEmbedding = TextEmbedding(
            torch.eye(24), dropout=hdnDropout, freeze=True, name='amEmbedding'
        ).to(device)
        self.pCNN = TextCNN(24, 64, [25], ln=True, name='pCNN').to(device)
        self.pFcLinear = MLP(64, outSize, dropout=hdnDropout, bnEveryLayer=True, 
                            dpEveryLayer=True, outBn=True, outAct=True, outDp=True, 
                            name='pFcLinear').to(device)
        
        self.esm_proj = MLP(esm_dim, outSize, [outSize], dropout=hdnDropout, 
                           bnEveryLayer=True, name='esm_proj').to(device)
        
        self.dCNN = TextCNN(75, 64, [7], ln=True, name='dCNN').to(device)
        self.dFcLinear = MLP(64, outSize, dropout=hdnDropout, bnEveryLayer=True, 
                            dpEveryLayer=True, outBn=True, outAct=True, outDp=True, 
                            name='dFcLinear').to(device)
        
        self.fFcLinear = MLP(fSize, outSize, fHiddenSizeList, outAct=True, 
                            name='fFcLinear', dropout=hdnDropout, dpEveryLayer=True, 
                            outDp=True, bnEveryLayer=True, outBn=True).to(device)
        self.cFcLinear = MLP(cSize, outSize, cHiddenSizeList, outAct=True, 
                            name='cFcLinear', dropout=hdnDropout, dpEveryLayer=True, 
                            outDp=True, bnEveryLayer=True, outBn=True).to(device)
        
        self.nodeGCN = GCN(outSize, outSize, gcnHiddenSizeList, name='nodeGCN', 
                          dropout=hdnDropout, dpEveryLayer=True, outDp=True, 
                          bnEveryLayer=True, outBn=True, resnet=resnet).to(device)
        
        self.fcLinear = MLP(outSize, 1, fcHiddenSizeList, dropout=fcDropout, 
                           bnEveryLayer=True, dpEveryLayer=True, outAct=False).to(device)
        
        self.criterion = nn.MSELoss()
        
        self.moduleList = nn.ModuleList([
            self.nodeEmbedding, self.cFcLinear, self.fFcLinear, self.nodeGCN, 
            self.fcLinear, self.amEmbedding, self.pCNN, self.pFcLinear, 
            self.dCNN, self.dFcLinear, self.esm_proj
        ])
    
    def calculate_y_logit(self, X, mode='train'):
        # Protein features
        Xam = (self.cFcLinear(X['aminoCtr']).unsqueeze(1) 
               if self.useFeatures.get('kmers', True) else 0) + \
              (self.pFcLinear(self.pCNN(self.amEmbedding(X['aminoSeq']))).unsqueeze(1) 
               if self.useFeatures.get('pSeq', True) else 0)
        
        # Handle both pooled (1D) and sequence (2D) ESM embeddings
        if 'esm_emb' in X and X['esm_emb'] is not None:
            esm_input = X['esm_emb']
            if len(esm_input.shape) == 3:
                esm_agg = esm_input.mean(dim=1) if self.esm_agg == 'mean' else esm_input[:, 0]
            else:
                esm_agg = esm_input
            Xam = Xam + self.esm_proj(esm_agg).unsqueeze(1)
        
        # Drug features
        Xat = (self.fFcLinear(X['atomFin']).unsqueeze(1) 
               if self.useFeatures.get('FP', True) else 0) + \
              (self.dFcLinear(self.dCNN(X['atomFea'])).unsqueeze(1) 
               if self.useFeatures.get('dSeq', True) else 0)
        
        # Bridge mechanism
        if self.nodeNum > 0:
            node = self.nodeEmbedding.dropout2(
                self.nodeEmbedding.dropout1(self.nodeEmbedding.embedding.weight)
            ).repeat(len(Xat), 1, 1)
            node = torch.cat([Xam, Xat, node], dim=1)
            nodeDist = torch.sqrt(torch.sum(node**2, dim=2, keepdim=True) + 1e-8)
            cosNode = torch.matmul(node, node.transpose(1, 2)) / (nodeDist * nodeDist.transpose(1, 2) + 1e-8)
            cosNode[cosNode < 0] = 0
            cosNode[:, range(node.shape[1]), range(node.shape[1])] = 1
            
            D = torch.eye(node.shape[1], dtype=torch.float32, device=self.device).repeat(len(Xam), 1, 1)
            D[:, range(node.shape[1]), range(node.shape[1])] = 1 / (torch.sum(cosNode, dim=2)**0.5)
            pL = torch.matmul(torch.matmul(D, cosNode), D)
            node_gcned = self.nodeGCN(node, pL)
            
            # DEBUG: Print shapes
            if not hasattr(self, 'debug_printed'):
                print(f"DEBUG: node_gcned shape: {node_gcned.shape}")
                print(f"DEBUG: outSize: {self.outSize}")
                self.debug_printed = True
            
            node_embed = node_gcned[:, 0, :] * node_gcned[:, 1, :]
            
            # DEBUG: Verify node_embed dimension
            if not hasattr(self, 'debug_printed2'):
                print(f"DEBUG: node_embed shape: {node_embed.shape}")
                self.debug_printed2 = True
        else:
            node_embed = (Xam * Xat).squeeze(dim=1)
            
        return self.fcLinear(node_embed).squeeze(dim=1)
    
    def forward(self, X):
        return self.calculate_y_logit(X)


# ==========================================
# 3. DATA LOADING
# ==========================================

def smiles_to_features(smiles, max_len=128):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None or mol.GetNumAtoms() == 0:
            return (np.zeros((max_len, 75), dtype=np.float32), 
                    np.zeros((1024,), dtype=np.float32), 
                    np.eye(max_len, dtype=np.float32), 
                    0)
        
        atom_features = []
        for atom in mol.GetAtoms():
            feat = [0] * 75
            atomic_num = atom.GetAtomicNum()
            feat[min(atomic_num, 74)] = 1
            feat[min(atom.GetDegree() + 25, 74)] = 1
            feat[min(abs(atom.GetFormalCharge()) + 35, 74)] = 1
            feat[min(int(atom.GetHybridization()) + 45, 74)] = 1
            feat[55 + int(atom.GetIsAromatic())] = 1
            atom_features.append(feat)
        
        if len(atom_features) < max_len:
            atom_features += [[0]*75] * (max_len - len(atom_features))
        else:
            atom_features = atom_features[:max_len]
        
        fp = np.zeros((1024,), dtype=np.float32)
        try:
            DataStructs.ConvertToNumpyArray(
                AllChem.GetHashedMorganFingerprint(mol, 2, nBits=1024), fp
            )
        except:
            pass
        
        num_atoms = mol.GetNumAtoms()
        adj = np.zeros((max_len, max_len), dtype=np.float32)
        if num_atoms > 0:
            adj_mol = Chem.rdmolops.GetAdjacencyMatrix(mol).astype(np.float32)
            actual_size = min(num_atoms, max_len)
            adj[:actual_size, :actual_size] = adj_mol[:actual_size, :actual_size]
        
        d = np.sum(adj, axis=0)
        d[d == 0] = 1
        d_inv_sqrt = np.power(d, -0.5)
        d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0
        d_mat_inv_sqrt = np.diag(d_inv_sqrt)
        adj_norm = d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt
        np.fill_diagonal(adj_norm, 1.0)
        
        return np.array(atom_features, dtype=np.float32), fp, adj_norm, num_atoms
        
    except Exception as e:
        print(f"Warning: Error processing SMILES '{smiles}': {e}")
        return (np.zeros((max_len, 75), dtype=np.float32), 
                np.zeros((1024,), dtype=np.float32), 
                np.eye(max_len, dtype=np.float32), 
                0)


def load_esm_embeddings(esm_path, protein_ids):
    print(f"Loading ESM embeddings from {esm_path}...")
    
    if not os.path.exists(esm_path):
        print(f"Warning: ESM file not found. Using random initialization.")
        return None, CONFIG['esm_dim'], 'pooled'
    
    try:
        data = np.load(esm_path, allow_pickle=True)
        if data.shape == () and data.dtype == object:
            esm_dict = data.item()
        else:
            raise ValueError("Unexpected ESM file format")
    except Exception as e:
        print(f"Error loading ESM: {e}. Using random initialization.")
        return None, CONFIG['esm_dim'], 'pooled'
    
    filtered = {}
    for pid in protein_ids:
        pid_str = str(pid)
        if pid_str in esm_dict and esm_dict[pid_str] is not None:
            emb = esm_dict[pid_str]
            if isinstance(emb, np.ndarray):
                filtered[pid_str] = torch.from_numpy(emb).float()
            else:
                filtered[pid_str] = torch.tensor(emb, dtype=torch.float)
    
    print(f"Loaded {len(filtered)}/{len(protein_ids)} ESM embeddings")
    
    if len(filtered) > 0:
        first_emb = next(iter(filtered.values()))
        print(f"First ESM embedding shape: {first_emb.shape}")
        if len(first_emb.shape) == 1:
            esm_dim = first_emb.shape[0]
            esm_type = 'pooled'
        else:
            esm_dim = first_emb.shape[1]
            esm_type = 'sequence'
        print(f"Auto-detected ESM dimension: {esm_dim}, type: {esm_type}")
    else:
        esm_dim = CONFIG['esm_dim']
        esm_type = 'pooled'
    
    return filtered, esm_dim, esm_type


class BridgeDTIDataset(Dataset):
    def __init__(self, df, smiles_col, prot_col, target_col, 
                 esm_dict, default_esm, scaler=None, fit_scaler=False,
                 pSeqMaxLen=1024, dSeqMaxLen=128, kmer_vectorizer=None):
        self.df = df.reset_index(drop=True)
        self.smiles_col = smiles_col
        self.prot_col = prot_col
        self.target_col = target_col
        self.esm_dict = esm_dict
        self.default_esm = default_esm
        self.scaler = scaler
        self.fit_scaler = fit_scaler
        self.pSeqMaxLen = pSeqMaxLen
        self.dSeqMaxLen = dSeqMaxLen
        self.kmer_vectorizer = kmer_vectorizer
        
        print("Pre-processing SMILES...")
        self.valid_indices = []
        for idx in range(len(self.df)):
            smiles = str(self.df.iloc[idx][self.smiles_col])
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None and mol.GetNumAtoms() > 0:
                self.valid_indices.append(idx)
        print(f"Valid samples: {len(self.valid_indices)}/{len(self.df)}")
        
        self._build_kmer_vectorizer()
        
    def _build_kmer_vectorizer(self):
        if self.kmer_vectorizer is None:
            self.ctr = CountVectorizer(ngram_range=(1, 3), analyzer='char')
            sample_seqs = self.df[self.prot_col].astype(str).head(1000).tolist()
            self.ctr.fit(sample_seqs)
            self.kmer_vectorizer = self.ctr
            print(f"NEW CountVectorizer vocabulary size: {len(self.ctr.get_feature_names_out())}")
        else:
            self.ctr = self.kmer_vectorizer
            print(f"USING SHARED CountVectorizer vocabulary size: {len(self.ctr.get_feature_names_out())}")
        
    def _tokenize_protein(self, seq):
        aa_vocab = {"<UNK>": 0, "<EOS>": 1}
        for aa in "ACDEFGHIKLMNPQRSTVWY":
            if aa not in aa_vocab:
                aa_vocab[aa] = len(aa_vocab)
        
        tokens = [aa_vocab.get(aa, 0) for aa in str(seq)[:self.pSeqMaxLen]]
        tokens += [1] * max(0, self.pSeqMaxLen - len(tokens))
        return tokens
    
    def _tokenize_drug(self, smiles):
        atom_vocab = {"<UNK>": 0, "<EOS>": 1}
        for symbol in "CNOcHnSBrIPFCl":
            if symbol not in atom_vocab:
                atom_vocab[symbol] = len(atom_vocab)
        
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return [0] * self.dSeqMaxLen
            
            tokens = []
            for atom in mol.GetAtoms()[:self.dSeqMaxLen]:
                symbol = atom.GetSymbol()
                tokens.append(atom_vocab.get(symbol, 0))
            tokens += [1] * max(0, self.dSeqMaxLen - len(tokens))
            return tokens
        except:
            return [0] * self.dSeqMaxLen
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        
        smiles = str(row[self.smiles_col])
        protein_seq = str(row[self.prot_col])
        target = float(row[self.target_col])
        
        if self.scaler is not None:
            if self.fit_scaler:
                target = self.scaler.fit_transform([[target]])[0, 0]
            else:
                target = self.scaler.transform([[target]])[0, 0]
        
        aminoSeq = self._tokenize_protein(protein_seq)
        aminoCtr = self.ctr.transform([protein_seq]).toarray().astype(np.float32)[0]
        
        atomFea, atomFin, atomGra, atom_count = smiles_to_features(smiles, self.dSeqMaxLen)
        atomSeq = self._tokenize_drug(smiles)
        
        pid = str(row[self.prot_col])
        esm_emb = self.esm_dict.get(pid, self.default_esm) if self.esm_dict else self.default_esm
        
        return {
            'aminoSeq': torch.tensor(aminoSeq, dtype=torch.long),
            'aminoCtr': torch.tensor(aminoCtr, dtype=torch.float32),
            'atomFea': torch.tensor(atomFea, dtype=torch.float32),
            'atomFin': torch.tensor(atomFin, dtype=torch.float32),
            'atomGra': torch.tensor(atomGra, dtype=torch.float32),
            'atomSeq': torch.tensor(atomSeq, dtype=torch.long),
            'esm_emb': esm_emb,
            'target': torch.tensor([target], dtype=torch.float32)
        }


def collate_fn(batch):
    aminoSeq = torch.stack([b['aminoSeq'] for b in batch])
    aminoCtr = torch.stack([b['aminoCtr'] for b in batch])
    atomFea = torch.stack([b['atomFea'] for b in batch])
    atomFin = torch.stack([b['atomFin'] for b in batch])
    atomGra = torch.stack([b['atomGra'] for b in batch])
    atomSeq = torch.stack([b['atomSeq'] for b in batch])
    
    first_esm = batch[0]['esm_emb']
    if len(first_esm.shape) == 1:
        esm_emb = torch.stack([b['esm_emb'] for b in batch])
    else:
        esm_emb = torch.stack([b['esm_emb'].mean(dim=0) for b in batch])
    
    targets = torch.cat([b['target'] for b in batch])
    
    return {
        'aminoSeq': aminoSeq,
        'aminoCtr': aminoCtr,
        'atomFea': atomFea,
        'atomFin': atomFin,
        'atomGra': atomGra,
        'atomSeq': atomSeq,
        'esm_emb': esm_emb,
    }, targets


# ==========================================
# 4. TRAINING FUNCTIONS
# ==========================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, count = 0, 0
    
    for X, y in loader:
        X = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in X.items()}
        y = y.to(device)
        
        optimizer.zero_grad()
        out = model(X)
        loss = model.criterion(out, y)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=20, norm_type=2)
        optimizer.step()
        
        total_loss += loss.item() * y.size(0)
        count += y.size(0)
        
        if device == 'cuda':
            torch.cuda.empty_cache()
    
    return total_loss / count


def evaluate(model, loader, device, scaler=None):
    model.eval()
    total_loss, count = 0, 0
    preds, tgts = [], []
    
    with torch.no_grad():
        for X, y in loader:
            X = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in X.items()}
            y = y.to(device)
            
            out = model(X)
            loss = model.criterion(out, y)
            
            total_loss += loss.item() * y.size(0)
            preds.append(out.cpu())
            tgts.append(y.cpu())
            count += y.size(0)
            
            if device == 'cuda':
                torch.cuda.empty_cache()
    
    p_np = torch.cat(preds).numpy().flatten()
    t_np = torch.cat(tgts).numpy().flatten()
    
    if scaler is not None:
        p_np = scaler.inverse_transform(p_np.reshape(-1, 1)).flatten()
        t_np = scaler.inverse_transform(t_np.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(t_np, p_np))
    mae = mean_absolute_error(t_np, p_np)
    r2 = r2_score(t_np, p_np)
    
    return total_loss / count, rmse, mae, r2


# ==========================================
# 5. LOGGING FUNCTION
# ==========================================

def log_experiment(model_name, iterations, learning_rate, depth,
                   train_mae, train_r2, train_rmse,
                   test_mae, test_r2, test_rmse, notes=""):
    log_file = CONFIG['log_file']
    
    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_name": model_name,
        "iterations": iterations,
        "learning_rate": learning_rate,
        "depth": depth,
        "train_mae": round(train_mae, 4),
        "train_r2": round(train_r2, 4),
        "train_rmse": round(train_rmse, 4),
        "test_mae": round(test_mae, 4),
        "test_r2": round(test_r2, 4),
        "test_rmse": round(test_rmse, 4),
        "notes": notes
    }
    
    if os.path.exists(log_file):
        log_df = pd.read_csv(log_file)
    else:
        log_df = pd.DataFrame()
    
    log_df = pd.concat([log_df, pd.DataFrame([log_entry])], ignore_index=True)
    log_df.to_csv(log_file, index=False)
    print(f"Logged: {model_name}")


# ==========================================
# 6. MAIN TRAINING PIPELINE
# ==========================================

def main():
    print("\n=== LOADING DATA ===")
    
    if not os.path.exists(CONFIG['data_path']):
        print(f"Error: Dataset not found at {CONFIG['data_path']}")
        return None
    
    df = pd.read_csv(CONFIG['data_path'])
    df = df.dropna(subset=[CONFIG['smiles_column'], CONFIG['protein_id_column'], CONFIG['target_column']])
    print(f"Loaded {len(df)} samples")
    
    unique_proteins = df[CONFIG['protein_id_column']].astype(str).unique().tolist()
    
    esm_dict, esm_dim, esm_type = load_esm_embeddings(CONFIG['esm_path'], unique_proteins)
    CONFIG['esm_dim'] = esm_dim
    default_esm = torch.mean(torch.stack(list(esm_dict.values())), dim=0) if esm_dict else torch.randn(esm_dim)
    print(f"ESM dimension: {esm_dim}, type: {esm_type}")
    print(f"default_esm shape: {default_esm.shape}")
    
    print("Applying StandardScaler to target...")
    scaler = StandardScaler()
    scaler.fit(df[[CONFIG['target_column']]])
    
    if CONFIG['cold_protein_split']:
        proteins = df[CONFIG['protein_id_column']].unique()
        train_prots, temp_prots = train_test_split(proteins, test_size=0.2, random_state=42)
        val_prots, test_prots = train_test_split(temp_prots, test_size=0.5, random_state=42)
        
        train_df = df[df[CONFIG['protein_id_column']].isin(train_prots)]
        val_df = df[df[CONFIG['protein_id_column']].isin(val_prots)]
        test_df = df[df[CONFIG['protein_id_column']].isin(test_prots)]
    else:
        train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    print(f"Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
    
    print("\n=== CREATING DATASETS ===")
    print("Creating TRAIN dataset...")
    train_ds = BridgeDTIDataset(
        train_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], 
        CONFIG['target_column'], esm_dict, default_esm, scaler=scaler, fit_scaler=False,
        kmer_vectorizer=None
    )
    
    CONFIG['cSize'] = len(train_ds.ctr.get_feature_names_out())
    print(f"\nCONFIG['cSize'] from train dataset: {CONFIG['cSize']}")
    
    sample_item = train_ds[0]
    aminoCtr_shape = sample_item['aminoCtr'].shape[0]
    esm_emb_shape = sample_item['esm_emb'].shape
    print(f"Sample aminoCtr shape: {aminoCtr_shape}")
    print(f"Sample esm_emb shape: {esm_emb_shape}")
    
    if CONFIG['cSize'] != aminoCtr_shape:
        print(f"ERROR: cSize mismatch!")
        return None
    else:
        print("✓ Train cSize verification passed!")
    
    print("\nCreating VAL dataset...")
    val_ds = BridgeDTIDataset(
        val_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], 
        CONFIG['target_column'], esm_dict, default_esm, scaler=scaler, fit_scaler=False,
        kmer_vectorizer=train_ds.ctr
    )
    
    print("Creating TEST dataset...")
    test_ds = BridgeDTIDataset(
        test_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], 
        CONFIG['target_column'], esm_dict, default_esm, scaler=scaler, fit_scaler=False,
        kmer_vectorizer=train_ds.ctr
    )
    
    print("\nCreating DataLoaders...")
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, 
                             collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, 
                           collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, 
                            collate_fn=collate_fn, num_workers=0)
    
    print("Data ready")
    
    print("\n=== INITIALIZING MODEL ===")
    print(f"Using cSize={CONFIG['cSize']}, esm_dim={CONFIG['esm_dim']}, outSize={CONFIG['outSize']}")
    
    model = DTI_Bridge_Regression(
        outSize=CONFIG['outSize'],
        cHiddenSizeList=CONFIG['cHiddenSizeList'],
        fHiddenSizeList=CONFIG['fHiddenSizeList'],
        fSize=CONFIG['fSize'],
        cSize=CONFIG['cSize'],
        gcnHiddenSizeList=CONFIG['gcnHiddenSizeList'],  # NOW EMPTY []
        fcHiddenSizeList=CONFIG['fcHiddenSizeList'],
        nodeNum=CONFIG['nodeNum'],
        resnet=CONFIG['resnet'],
        hdnDropout=CONFIG['hdnDropout'],
        fcDropout=CONFIG['fcDropout'],
        device=torch.device(CONFIG['device']),
        esm_dim=CONFIG['esm_dim'],
        esm_agg=CONFIG['esm_agg']
    ).to(CONFIG['device'])
    
    print(f"\nModel cFcLinear first layer input size: {model.cFcLinear.inSize}")
    print(f"Model esm_proj first layer input size: {model.esm_proj.inSize}")
    print(f"Model nodeGCN inSize: {model.nodeGCN.inSize}, outSize: {model.nodeGCN.outSize}")
    print(f"Model fcLinear first layer input size: {model.fcLinear.inSize}")
    
    if model.cSize != CONFIG['cSize']:
        print(f"ERROR: Model cSize mismatch!")
        return None
    if model.esm_dim != CONFIG['esm_dim']:
        print(f"ERROR: Model esm_dim mismatch!")
        return None
    if model.outSize != CONFIG['outSize']:
        print(f"ERROR: Model outSize mismatch!")
        return None
    
    print("✓ Model dimension verification passed!")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    print("\n=== TESTING FORWARD PASS ===")
    model.eval()
    with torch.no_grad():
        test_batch = next(iter(train_loader))
        X_test, y_test = test_batch
        X_test = {k: v.to(CONFIG['device']) if isinstance(v, torch.Tensor) else v for k, v in X_test.items()}
        
        print(f"Input aminoCtr shape: {X_test['aminoCtr'].shape}")
        print(f"Input esm_emb shape: {X_test['esm_emb'].shape}")
        print(f"Expected cSize: {CONFIG['cSize']}")
        print(f"Expected esm_dim: {CONFIG['esm_dim']}")
        print(f"Expected outSize: {CONFIG['outSize']}")
        
        try:
            out_test = model(X_test)
            print(f"✓ Forward pass successful! Output shape: {out_test.shape}")
        except Exception as e:
            print(f"ERROR in forward pass: {e}")
            import traceback
            traceback.print_exc()
            return None
    model.train()
    
    print("\n=== STARTING TRAINING ===")
    best_val_rmse = float('inf')
    patience_counter = 0
    final_epoch = 0
    prev_lr = CONFIG['lr']
    
    train_losses, val_rmse_list = [], []
    
    for epoch in range(1, CONFIG['epochs'] + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, CONFIG['device'])
        val_loss, val_rmse, val_mae, val_r2 = evaluate(model, val_loader, CONFIG['device'], scaler=scaler)
        
        scheduler.step(val_rmse)
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr < prev_lr:
            print(f"   LR decreased: {prev_lr:.6f} -> {current_lr:.6f}")
            prev_lr = current_lr
        
        is_best = val_rmse < best_val_rmse
        if is_best:
            best_val_rmse = val_rmse
            patience_counter = 0
            torch.save(model.state_dict(), CONFIG['model_save_path'])
            print(f"New best model saved! (Val RMSE: {val_rmse:.4f})")
        else:
            patience_counter += 1
        
        t1 = time.time()
        train_losses.append(train_loss)
        val_rmse_list.append(val_rmse)
        
        print(f"Ep {epoch:03d} | LR: {current_lr:.1e} | Time: {t1-t0:.1f}s | "
              f"Train: {train_loss:.4f} | Val RMSE: {val_rmse:.4f} | "
              f"Best: {is_best} | Patience: {patience_counter}/{CONFIG['patience']}")
        
        gc.collect()
        if CONFIG['device'] == 'cuda':
            torch.cuda.empty_cache()
        
        if patience_counter >= CONFIG['patience']:
            print(f"\nEARLY STOPPING! No improvement for {CONFIG['patience']} epochs.")
            final_epoch = epoch
            break
        final_epoch = epoch
    
    print("Training completed")
    
    print("\n=== FINAL TESTING ===")
    if os.path.exists(CONFIG['model_save_path']):
        model.load_state_dict(torch.load(CONFIG['model_save_path'], weights_only=True))
        print("Loaded best model")
    
    test_loss, test_rmse, test_mae, test_r2 = evaluate(model, test_loader, CONFIG['device'], scaler=scaler)
    
    train_loss, train_rmse, train_mae, train_r2 = evaluate(model, train_loader, CONFIG['device'], scaler=scaler)
    
    print(f"FINAL RESULTS:")
    print(f"   Train RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, R2: {train_r2:.4f}")
    print(f"   Test  RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}, R2: {test_r2:.4f}")
    
    log_experiment(
        model_name="BridgeDTI_ESM_Regression",
        iterations=final_epoch,
        learning_rate=CONFIG['lr'],
        depth=len(CONFIG['gcnHiddenSizeList']) + len(CONFIG['fcHiddenSizeList']),
        train_mae=train_mae,
        train_r2=train_r2,
        train_rmse=train_rmse,
        test_mae=test_mae,
        test_r2=test_r2,
        test_rmse=test_rmse,
        notes=f"ESM_dim{CONFIG['esm_dim']}+esm_type{esm_type}+nodeNum{CONFIG['nodeNum']}+cSize{CONFIG['cSize']}+outSize{CONFIG['outSize']}"
    )
    
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(val_rmse_list, label='Val RMSE', color='orange', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('RMSE')
    plt.title('Validation RMSE')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(BASE_DIR, "results", "training_plot.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\nModel saved: {CONFIG['model_save_path']}")
    print(f"Log saved: {CONFIG['log_file']}")
    print(f"Plot saved: {plot_path}")
    
    return {
        'test_rmse': test_rmse,
        'test_mae': test_mae,
        'test_r2': test_r2,
        'best_val_rmse': best_val_rmse
    }


if __name__ == "__main__":
    results = main()
    if results is not None:
        print(f"\nSUMMARY: Test RMSE={results['test_rmse']:.4f}, Best Val RMSE={results['best_val_rmse']:.4f}")
    else:
        print("\nERROR: Training failed. Check the error messages above.")

All libraries imported successfully
Setting up paths...
Base project directory: /home/ubuntu/project
Device: cuda
GPU: NVIDIA A100 80GB PCIe

=== LOADING DATA ===
Loaded 94459 samples
Loading ESM embeddings from /home/ubuntu/project/cache/esm_embeddings.npy...
Loaded 451/452 ESM embeddings
First ESM embedding shape: torch.Size([320])
Auto-detected ESM dimension: 320, type: pooled
ESM dimension: 320, type: pooled
default_esm shape: torch.Size([320])
Applying StandardScaler to target...
Split: Train=75567, Val=9446, Test=9446

=== CREATING DATASETS ===
Creating TRAIN dataset...
Pre-processing SMILES...
Valid samples: 75567/75567
NEW CountVectorizer vocabulary size: 882

CONFIG['cSize'] from train dataset: 882
Sample aminoCtr shape: 882
Sample esm_emb shape: torch.Size([320])
✓ Train cSize verification passed!

Creating VAL dataset...
Pre-processing SMILES...
Valid samples: 9446/9446
USING SHARED CountVectorizer vocabulary size: 882
Creating TEST dataset...
Pre-processing SMILES...
Valid 